# Evidence Errors and Downstream Decisions

Which evidence or mapping errors change an action, and which worsen its quality? This notebook separates evidence interpretation, action selection, and consequence estimation. The included private-information voting scenario is an example, not a claim of domain-general validation.

The workflow illustrates an exact decision contract, inspects published frozen-model findings, and evaluates a small fresh sample. Install this repository with `python -m pip install -e ".[notebook]"` before running all cells. No external data or model download is needed afterward.

In [ ]:
from datetime import datetime, timezone
from fractions import Fraction
from html import escape
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import HTML, display

from decision_criticality import evaluate_decision
from decision_criticality.artifacts import ASSETS, FAMILIES, SEEDS, load_json
from decision_criticality.example import generate_instance, posterior, reference_posterior
from decision_criticality.experiment import run_study
from decision_criticality.learning import run_training

torch.set_num_threads(1)
candidates = (Path.cwd(), Path.cwd().parent, Path.cwd() / "decision_critical_graph_study")
ROOT = next((path for path in candidates if (path / "decision_criticality").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Run the notebook from this repository or its parent directory.")

GAMES_PER_SEED = 3
DATA_SEEDS = (41101, 42101, 43101)
BOOTSTRAP_SAMPLES = 1000
RETRAIN = False
TRAINING_EPOCHS = 10
OUTPUT = ROOT / "results" / ("notebook_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))


def show_table(headers, rows):
    header = "".join(f"<th>{escape(str(value))}</th>" for value in headers)
    body = "".join("<tr>" + "".join(f"<td>{escape(str(value))}</td>" for value in row) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))


print(f"Fresh sample: {len(DATA_SEEDS) * GAMES_PER_SEED} instances; {len(SEEDS)} fixed training seeds per family.")

## Decision Contract and Example

The executed action can be optimal under edited evidence yet lose value under the original evidence. If edited optima tie, the exact reference-loss range distinguishes possible from unavoidable harm. That range covers edited-optimal actions only, not arbitrary suboptimal choices.

The included example uses six players, uncertain report authorship, role-dependent reliability, private observations, and authenticated provenance. It is not standard Werewolf or a full-game strategy model. The exact posterior below is verified by an independent enumeration. Hidden assignments are not model inputs.

In [ ]:
contract = evaluate_decision(
    reference=(Fraction(3, 5), Fraction(2, 5)),
    edited=(Fraction(1, 2), Fraction(1, 2)),
    action=1,
    initial_action=0,
)
show_table(
    ("Quantity", "Value"),
    [(key, contract[key]) for key in (
        "reference_regret", "edited_regret", "incremental_reference_regret",
        "minimum_optimal_regret", "maximum_optimal_regret", "policy_dependent",
    )],
)

instance = generate_instance(np.random.default_rng(2026), "illustration")
observation = instance["views"][0]
exact = posterior(observation)
assert exact == reference_posterior(observation)
assert sum(exact) == 1
show_table(("Visible field", "Value"), [(key, observation[key]) for key in ("observer", "own_role", "inspection", "peer", "audit")])
show_table(("Origin", "Claimed source", "Target", "Status claim", "Forwarded"),
           [(item["root"], item["author"], item["target"], item["wolf_claim"], item["relay"]) for item in observation["reports"]])
figure, axis = plt.subplots(figsize=(7, 3))
axis.bar(range(6), [float(value) for value in exact], color="#207c78")
axis.set(xlabel="Player", ylabel="Exact posterior", title="Visible evidence and candidate probability")
figure.tight_layout()
plt.show()

## Published Frozen Confirmation

The published confirmation contains 120 independent instances and 28,200 paired cases. Four model families retain all three fixed training seeds. The enriched training arms receive matched near-tie exposure and background evidence; one repeats existing examples, while the other adds distinct independent evidence.

The table below reads the bundled reference artifact. Actual-action targets may differ across models. Common-action MAE holds the baseline's action and exact target fixed to isolate valuation. The reported primary improvement is in regret estimation, not demonstrated lower original-reference harm. Binary classification gains are mixed. Additional independent evidence does not establish a near-tie decision advantage over repetition.

In [ ]:
published = load_json(ASSETS / "reference.json")
fields = ("regret_mae", "common_action_absolute_error", "edited_posterior_regret", "own_policy_regret")
published_means = {
    family: [float(np.mean([published["metrics"][f"{family}_{seed}"]["all"][field] for seed in SEEDS])) for field in fields]
    for family in FAMILIES
}
print(f"Published sample: {published['games']} instances, {published['pairs']} pairs, {published['observations']} exact-checked observations.")
show_table(
    ("Family", "Actual-action MAE", "Common-action MAE", "Edited regret", "Reference regret"),
    [(family, *(f"{value:.6f}" for value in published_means[family])) for family in FAMILIES],
)
primary = next(row for row in published["paired_intervals"] if row["scope"] == "all" and
               row["contrast"] == "independent_minus_baseline" and row["metric"] == "regret_absolute_error")
show_table(("Primary contrast", "Difference", "95% interval"), [
    ("Independent minus baseline regret MAE", f"{primary['difference']:.6f}", f"[{primary['low']:.6f}, {primary['high']:.6f}]")
])
figure, axes = plt.subplots(1, 2, figsize=(10, 3.5))
colors = ("#66756b", "#b57b29", "#487eb0", "#207c78")
for axis, column, title in zip(axes, (0, 3), ("Regret-estimation error", "Original-reference decision loss")):
    axis.bar(FAMILIES, [published_means[family][column] for family in FAMILIES], color=colors)
    axis.set(title=title, ylabel="Case-weighted seed mean")
    axis.tick_params(axis="x", labelrotation=15)
figure.tight_layout()
plt.show()

## Fresh Evaluation

This small run demonstrates the workflow; it is not the published 120-instance confirmation and is not sufficient for fine-grained claims. It uses bundled frozen models and validation-selected thresholds, rejects recorded prior-instance duplicates, independently checks exact posteriors, and saves replayable artifacts under `results/`.

Set `RETRAIN = True` in the settings cell to refit all matched controls from the saved training evidence before evaluating fresh instances. That optional mode uses validation-only checkpoint selection and recalibrates thresholds before test generation. It does not rerun rare-example acquisition. For the full frozen replay, use `python -m decision_criticality --replay`.

In [ ]:
model_directory = None
if RETRAIN:
    model_directory, fits = run_training(OUTPUT / "models", epochs=TRAINING_EPOCHS)
    figure, axis = plt.subplots(figsize=(8, 3.5))
    for name, fit in fits.items():
        axis.plot([row["epoch"] for row in fit["history"]], [row["cross_entropy"] for row in fit["history"]], label=name)
    axis.set(xlabel="Epoch", ylabel="Validation cross entropy", title="Matched training controls")
    axis.legend(fontsize=7, ncol=3)
    figure.tight_layout()
    plt.show()

output, fresh = run_study(
    output_dir=OUTPUT / "evaluation",
    games_per_seed=GAMES_PER_SEED,
    data_seeds=DATA_SEEDS,
    bootstrap_samples=BOOTSTRAP_SAMPLES,
    model_dir=model_directory,
)
assert all(fresh["verification"].values())
show_table(("Verification", "Passed"), fresh["verification"].items())
print("Saved report:", (output / "report.md").relative_to(ROOT))

In [ ]:
show_table(
    ("Scope", "Metric", "Independent minus baseline", "95% interval", "Instances"),
    [(row["scope"], row["metric"], f"{row['difference']:.6f}", f"[{row['low']:.6f}, {row['high']:.6f}]", row["games"])
     for row in fresh["paired_intervals"] if row["scope"] == "all" and row["contrast"] == "independent_minus_baseline"],
)

edit_scopes = sorted(scope for scope in fresh["metrics"]["baseline_42"] if scope.startswith("edit:"))
figure, axes = plt.subplots(1, 2, figsize=(11, 4))
for axis, field, title in zip(axes, ("edited_regret", "reference_regret"), ("Loss under edited evidence", "Loss under reference evidence")):
    for family, color in zip(FAMILIES, colors):
        means = [np.mean([fresh["metrics"][f"{family}_{seed}"][scope][field] for seed in SEEDS]) for scope in edit_scopes]
        axis.plot([scope.split(":", 1)[1] for scope in edit_scopes], means, marker="o", label=family, color=color)
    axis.set(title=title, ylabel="Case-weighted seed mean")
    axis.tick_params(axis="x", labelrotation=40)
axes[0].legend(fontsize=8)
figure.tight_layout()
plt.show()

role_scopes = sorted(scope for scope in fresh["metrics"]["baseline_42"] if scope.startswith("role:"))
show_table(
    ("Family", "Observer role", "Regret MAE", "Edited regret", "Reference regret"),
    [(family, scope.split(":", 1)[1], *(
        f"{np.mean([fresh['metrics'][f'{family}_{seed}'][scope][field] for seed in SEEDS]):.6f}"
        for field in ("regret_error", "edited_regret", "reference_regret")))
     for family in FAMILIES for scope in role_scopes],
)

## Interpretation

- The relevant target is the consequence for the executed action. A mismatch with another tie policy is not automatically a valuation error.
- Edited-evidence optimality, reference-information loss, and realized hidden-world correctness are distinct quantities. Unchanged evidence can retain a model's pre-existing error without causing incremental damage.
- Exact possible/unavoidable bounds concern edited-optimal actions and are oracle diagnostics, not learned detectors.
- The published result supports improved consequence valuation and edited-evidence decisions, not demonstrated lower reference harm or a near-tie advantage of independent evidence over repetition.
- Whole-instance intervals condition on frozen models, acquired evidence, and validation thresholds. Training seeds are not additional independent test instances. The small fresh demonstration does not replace the published sample.
- Constructed contexts retain the assumed original signal channel, not natural-play or context-selection calibration. This example does not establish full-game strategy, universal relation necessity, or generalization to other domains.

See [ANALYSIS.md](ANALYSIS.md) for the full methods and uncertainty discussion. To explore another domain, replace the example's observations, edits, exact utilities, and encoding while keeping the decision contract explicit.